# 01 — Analyse exploratoire des données (EDA)

Jeu synthétique `data/synthetic/dataset.csv` (3 000 profils, seed 42).
Méthode de génération, hypothèses et biais : voir `data/synthetic/DONNEES-SYNTHETIQUES.md`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/synthetic/dataset.csv')
print(df.shape)
df.head()

In [ ]:
# Équilibre des classes : CAA domine (~14 %), IAA est rare (~1,6 %).
ax = df['filiere'].value_counts().sort_values().plot.barh(figsize=(7, 6), color='#1e6b45')
ax.set_title('Répartition des filières recommandées')
ax.set_xlabel('Nombre de profils')
plt.tight_layout(); plt.show()

In [ ]:
# Notes moyennes par filière : vérifie les corrélations attendues
# (maths élevées pour ISAIA, langues pour TEH/DTJA...).
notes = df.groupby('filiere')[['note_maths', 'note_sciences', 'note_langues', 'note_eco']].mean()
notes.round(2).sort_values('note_maths', ascending=False)

In [ ]:
# Intérêt dominant par filière (top 3) : cohérence profil <-> étiquette.
expl = df.assign(interet=df['interets'].str.split('|')).explode('interet')
top = (expl.groupby(['filiere', 'interet']).size().rename('n').reset_index()
         .sort_values(['filiere', 'n'], ascending=[True, False])
         .groupby('filiere').head(3))
top.pivot_table(index='filiere', columns='interet', values='n', fill_value=0)

In [ ]:
# Distribution des séries de bac et des environnements de travail.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df['serie_bac'].value_counts().plot.bar(ax=axes[0], color='#1e6b45', title='Séries de bac')
df['environnement'].value_counts().plot.bar(ax=axes[1], color='#a85d1c', title='Environnements')
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## Lectures

- **Déséquilibre de classes** réel mais gérable → `class_weight='balanced'` à l'entraînement.
- Les corrélations notes ↔ filières correspondent aux règles de génération documentées : le modèle
  apprendra principalement ces règles (limite nommée par le sujet), d'où la validation sur l'enquête réelle.
- Aucune valeur manquante ni aberrante par construction ; les réponses d'enquête, elles, devront être nettoyées.